# Solutions · 16 · Stochastic differential equations

Worked solutions to the exercises of [notebook 16](../notebooks/16_stochastic_differential_equations.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engstoch is missing): install it from GitHub
    import engstoch
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engstoch"], check=True)
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engstoch import sde, brownian as bm
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

R = np.random.default_rng      # R(seed): a fresh, reproducible generator

## Exercise 1

For the Langevin equation dX = -X dt + sqrt(2) dW, compare the stationary variance of Euler-Maruyama with step h against the exact value 1. Derive the Euler value 1/(1 - h/2) and check it.

In [2]:
rows = []
for h in (0.5, 0.2, 0.1, 0.02):
    X = sde.euler_maruyama(lambda t, x: -x, lambda t, x: np.sqrt(2.0) + 0 * x, 0.0, 200.0, int(200 / h), 400, R(int(100 * h)))["X"]
    rows.append((h, X[:, int(20 / h):].var(), 1 / (1 - h / 2)))
print(pd.DataFrame(rows, columns=["h", "simulated stationary variance", "1/(1 - h/2)"]).to_string(index=False, float_format="%.4f"))

     h  simulated stationary variance  1/(1 - h/2)
0.5000                         1.3435       1.3333
0.2000                         1.1077       1.1111
0.1000                         1.0436       1.0526
0.0200                         1.0056       1.0101


Euler gives the AR(1) X_{k+1} = (1 − h)X_k + √(2h) Z with stationary variance 2h / (1 − (1 − h)²) =
1/(1 − h/2): the error in the *invariant measure* is first order in h, which matters for long-time simulation
(molecular dynamics, MCMC by Langevin dynamics, where a Metropolis correction removes it).

## Exercise 2

Simulate the CIR process with 2 kappa theta < sigma^2 by plain Euler (with max(x, 0) inside the square root only) and by full truncation; compare their means at T = 1 with the exact value for h = 0.1, 0.01.

In [3]:
x0, kappa, theta, sigma, T = 0.04, 0.5, 0.04, 0.4, 1.0          # 2 kappa theta = 0.04 < sigma^2 = 0.16
exact = float(sde.cir_mean(x0, kappa, theta, T))
for h in (0.1, 0.01):
    n = int(T / h); g = R(int(1 / h))
    x = np.full(200_000, x0)
    for k in range(n):
        x = x + kappa * (theta - x) * h + sigma * np.sqrt(np.maximum(x, 0) * h) * g.standard_normal(len(x))
    ft = sde.cir_full_truncation(x0, kappa, theta, sigma, T, n, 200_000, R(int(1 / h) + 7))[:, -1]
    print(f"h = {h}: absorption-style Euler mean {x.mean():.5f} (negative values {np.mean(x < 0):.3f}), full truncation {ft.mean():.5f}, exact {exact:.5f}")

h = 0.1: absorption-style Euler mean 0.03984 (negative values 0.246), full truncation 0.04135, exact 0.04000


h = 0.01: absorption-style Euler mean 0.04004 (negative values 0.137), full truncation 0.04008, exact 0.04000


When the Feller condition fails, zero is reachable and Euler steps overshoot below it. With max(x, 0) only
inside the square root, a quarter of the paths are negative at h = 0.1 - impossible values for a rate or a
variance. Its *mean* happens to be close here (negative excursions and the missing noise near zero partly
cancel), while full truncation is biased upwards at h = 0.1; both biases vanish as h shrinks. The case for full
truncation is not the mean but positivity and robustness: in the Heston model the CIR value is a variance whose
square root drives the price, and a negative value breaks the simulation.

## Exercise 3

**Implement it yourself:** write the MLMC level estimator for the GBM call (coupled coarse and fine Euler paths) and verify that Var(P_l - P_{l-1}) halves from one level to the next.

In [4]:
def level_pair(level, n, g, S0=100.0, r=0.05, sig=0.2, T=1.0):
    nf = 2**level; hf = T / nf
    dW = np.sqrt(hf) * g.standard_normal((n, nf))
    Sf = np.full(n, S0)
    for k in range(nf): Sf = Sf * (1 + r * hf + sig * dW[:, k])
    if level == 0: return np.exp(-r * T) * np.maximum(Sf - 100, 0)
    Sc = np.full(n, S0); dWc = dW[:, 0::2] + dW[:, 1::2]
    for k in range(nf // 2): Sc = Sc * (1 + r * 2 * hf + sig * dWc[:, k])
    return np.exp(-r * T) * (np.maximum(Sf - 100, 0) - np.maximum(Sc - 100, 0))
v = [np.var(level_pair(l, 100_000, R(l))) for l in range(1, 9)]
print("Var(P_l - P_l-1), l = 1..8:", np.round(v, 5).tolist())
print("ratios V_l / V_l+1:", np.round(np.array(v[:-1]) / np.array(v[1:]), 2).tolist())

Var(P_l - P_l-1), l = 1..8: [2.39239, 1.27694, 0.67597, 0.35008, 0.17908, 0.09068, 0.04561, 0.02296]
ratios V_l / V_l+1: [1.87, 1.89, 1.93, 1.95, 1.97, 1.99, 1.99]


Driving the coarse path with the sums of pairs of fine increments couples the two; their payoffs differ by
O(√h) (the strong error), so the variance of the difference is O(h) and halves from level to level. That decay
is what lets MLMC spend most of its samples on the cheap levels.